## **AdamW optimizer**

### 1. What is AdamW? The Core Idea

**AdamW (Adam with Weight Decay)** is a variant of the Adam optimizer that fixes a critical flaw in the original Adam algorithm's implementation of **L2 regularization** (often referred to as "weight decay").

The key insight is that **L2 regularization and weight decay are not identical for adaptive gradient algorithms like Adam.** AdamW decouples weight decay from the gradient update, applying it directly to the weights *after* controlling for the adaptive learning rate.

**The Problem in One Sentence:**

Original Adam incorrectly mixes adaptive learning rates with L2 regularization. AdamW fixes this by decoupling true weight decay from the gradient-based update.

---

### 2. The Flaw in Original Adam (and Adam+L2)

To understand AdamW, we must first see the problem it solves.

#### **Standard L2 Regularization (as used in SGD)**

In SGD, when you add L2 regularization, the update rule is:
`w = w - lr * (dw + λ * w)`

where:

*   `dw` is the gradient of the loss w.r.t. `w` (`dL/dw`)
*   `λ` is the weight decay coefficient (`weight_decay`)

This is equivalent to **true weight decay**: `w = w * (1 - lr * λ) - lr * dw`. The decay is proportional to the learning rate `lr`.


#### **L2 Regularization in Original Adam**

Adam, and other adaptive optimizers, incorporate L2 regularization by adding `λ * w` to the gradient `dw` *before* computing the adaptive moments (mean `m_t` and variance `v_t`).

The update becomes:

1.  `g_t = dw + λ * w`  **(L2 is added to the gradient)**
2.  `m_t = β1 * m_{t-1} + (1 - β1) * g_t`
3.  `v_t = β2 * v_{t-1} + (1 - β2) * g_t^2`
4.  `w = w - lr * m_t / (sqrt(v_t) + ε)`

**Why this is a problem:**

Because `g_t` includes `λ * w`, the adaptive learning rate calculation (`m_t / sqrt(v_t)`) is applied not just to the loss gradient `dw`, but also to the weight decay term. This means the **effective strength of the regularization depends on the adaptive learning rate** for each parameter, which changes over time. This is not true weight decay and often leads to suboptimal regularization, especially when fine-tuning pre-trained models.

---

### 3. How AdamW Fixes It: Decoupled Weight Decay

The authors of the paper ["Decoupled Weight Decay Regularization" (Loshchilov & Hutter, 2017)](https://arxiv.org/abs/1711.05101) proposed a simple but powerful fix: **decouple the weight decay term from the adaptive gradient update.**

The AdamW algorithm does the following:
1.  Calculate the adaptive moments using *only the raw loss gradient* `dw`.
2.  Perform the standard Adam update using these moments.
3.  **Then, subtract the weight decay term *separately*.**

The update rule becomes:
1.  `m_t = β1 * m_{t-1} + (1 - β1) * dw`  **(Notice: `g_t = dw`, no L2 here!)**
2.  `v_t = β2 * v_{t-1} + (1 - β2) * dw^2`
3.  `w = w - lr * (m_t / (sqrt(v_t) + ε) + λ * w)`  **(Weight decay is added *after*)**

This is now equivalent to true weight decay, as the decay term is no longer scaled by the adaptive learning rate. It's a fixed, consistent shrinkage of the weights towards zero at each step, independent of the gradient-based update.

**Analogy:**
*   **Adam + L2:** Mixing salt (weight decay) into your flour (gradients) before baking. The final taste depends on the whole recipe.
*   **AdamW:** Sprinkling salt on top of the finished dish. You control the saltiness directly and consistently.

---

### 4. Why is AdamW the Gold Standard for Transformers/LLMs?

AdamW isn't just a minor improvement; it's a critical component for modern deep learning success.

1.  **Superior Regularization:** Transformers and LLMs are highly over-parameterized and prone to overfitting. Effective regularization is paramount. AdamW provides a more stable and predictable form of regularization than Adam+L2, leading to better generalization performance.

2.  **Stable Fine-Tuning:** The field of NLP heavily relies on fine-tuning pre-trained models (like BERT, GPT). These models are sensitive to the hyperparameter `weight_decay`. AdamW makes the effect of the `weight_decay` hyperparameter consistent and easier to tune, as its effect is no longer entangled with the adaptive learning rates.

3.  **Empirical Dominance:** Nearly every seminal Transformer-based model from BERT onwards has been trained using AdamW. Its performance advantages are well-documented and reproducible across a vast range of tasks.
    *   **BERT:** "We use AdamW [...] with weight decay regularization."
    *   **GPT-2/GPT-3:** Use AdamW (or a variant very close to it).
    *   **RoBERTa, T5, etc.:** All use AdamW.

4.  **Better Convergence:** Research and practice have shown that AdamW often converges faster and to a better minimum than Adam with L2 regularization, especially on large-scale problems.

---

### 5. Practical Implementation and Code

Using AdamW is straightforward in all major frameworks. The key is to use the dedicated `AdamW` optimizer and **not** to use the `weight_decay` parameter in the standard `Adam` optimizer.

**PyTorch:**

```python
import torch
from torch.optim import AdamW

# Model and hyperparameters
model = MyTransformerModel()
learning_rate = 5e-5
weight_decay = 0.01  # A very common and good default for fine-tuning

# IMPORTANT: Use the dedicated AdamW optimizer
optimizer = AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,  # This is the CORRECT way to specify decay
    betas=(0.9, 0.999),         # Standard beta values for Adam
    eps=1e-8                    # Standard epsilon value
)

# The training loop remains identical
for batch in dataloader:
    optimizer.zero_grad()
    loss = model(batch)
    loss.backward()
    optimizer.step() # The decoupled weight decay happens here
```


**Hugging Face Transformers:**

The `Trainer` API uses AdamW by default.

```python
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    # AdamW-specific hyperparameters:
    weight_decay=0.01,           # The decoupled weight decay
    adam_beta1=0.9,              # Beta1
    adam_beta2=0.999,            # Beta2
    adam_epsilon=1e-8,           # Epsilon
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)
trainer.train() # Uses AdamW under the hood
```

---

### 6. Key Hyperparameters and Tuning Tips

1.  **`weight_decay`**: The most important new hyperparameter.
    *   **Common Range:** Typically between `0.0` (no decay) and `0.1`.
    *   **Strong Default:** **`0.01`** is an excellent starting point for most tasks, especially fine-tuning.
    *   **Tuning:** If the model is underfitting, try a lower value (e.g., `0.005` or `0.0`). If it's overfitting, try a higher value (e.g., `0.05`).


2.  **`learning_rate` (`lr`)**: Works in tandem with `weight_decay`. Because decay is now decoupled, these two hyperparameters are more orthogonal and easier to tune separately. Standard LR ranges for AdamW in Transformers are typically `1e-5` to `5e-5` for fine-tuning and `1e-4` to `1e-3` for pre-training from scratch.


3.  **`betas`**: Controls the momentum and adaptive learning rate. `(0.9, 0.999)` is almost universally used and rarely needs changing.


4.  **`eps` (`epsilon`)**: A numerical stability constant. `1e-8` is the standard and should not be changed unless you have a very specific reason.



### Summary

| Aspect | Key Takeaway |
| :--- | :--- |
| **Core Idea** | Fixes Adam by decoupling true weight decay from the adaptive gradient update. |
| **Problem Solved** | In Adam, L2 regularization is scaled by the adaptive learning rate, weakening its effect. |
| **How it Works** | Applies weight decay directly to the weights *after* the Adam update, independent of the adaptive LR. |
| **Why for LLMs** | Provides stable and effective regularization, crucial for large, over-parameterized models. It's the empirical standard. |
| **Implementation** | Use a dedicated `AdamW` optimizer. **Do not** use `weight_decay` in standard `Adam`. |
| **Key Hyperparameter** | `weight_decay` (start with `0.01`). `learning_rate` is also critical. |


In conclusion, `AdamW` is a fundamental advancement in optimization that provides more predictable and effective regularization. Its adoption is a key reason behind the successful and stable training of the massive Transformer models that power today's AI applications. If you are training a modern deep learning model, especially in NLP, `AdamW` should be your default optimizer choice.

---